# Level マップ — カメラ位置取得

**前提**: UE Editor で `/Game/Maps/Level` を開き **PIE 実行中** にこのノートを実行してください。

## 目的

ブロック設置領域の **2 角** を決めるため、PIE 中のカメラ位置（UE 世界座標）を取得します。

1. 角 A の見た目に合わせてビュー／カメラを移動 → **セル 3** を実行
2. 角 B も同様 → 再度 **セル 3**
3. 表示された `loc_m=(x, y, z)` をメモしてチャットに渡してください（XY が領域の角、Z は参考）

補助: `id=0`（PawnSensor 等）がプレイヤー視点のことが多いです。別カメラを動かしている場合はその `id` を使ってください。

カーネル: `conda activate simworld`

In [1]:
import importlib
import sys
from pathlib import Path
from typing import Optional

from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_level_dir = _root / "dev" / "grid_env_level_semantic"
_g10k_dir = _root / "dev" / "grid_env_10k"
_geh_dir = _root / "dev" / "grid_env_hri"
for p in (_root, _level_dir, _g10k_dir, _geh_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

ucv: Optional[UnrealCV] = None
print(f"[Paths] root={_root}")
print(f"[Paths] level_semantic={_level_dir}")

[Paths] root=/home/winder17wsl_ishizawalab/00_kotaprivate/Program/SimWorld
[Paths] level_semantic=/home/winder17wsl_ishizawalab/00_kotaprivate/Program/SimWorld/dev/grid_env_level_semantic


In [2]:
import level_camera_probe as lcp

importlib.reload(lcp)

if not lcp.wait_for_ue_port(90.0):
    raise RuntimeError("UnrealCV port 9000 not reachable — open Level map and start PIE.")

ucv, _ = lcp.ensure_connection()
if not ucv.client.isconnected():
    raise RuntimeError("UnrealCV not connected.")
print("OK: UnrealCV connected (Level PIE)")

[UE] Probing UnrealCV on ['127.0.0.1', '10.103.0.1', '10.255.255.254'] (timeout=3s each) ...
[UE] probe OK: 127.0.0.1:9000


INFO:__init__:238:Got connection confirm: b'connected to SimWorld'


[UE] probe TCP failed 10.103.0.1:9000 — timed out
[UE] probe skip: 10.103.0.1:9000
[UE] probe TCP failed 10.255.255.254:9000 — [Errno 111] Connection refused
[UE] probe skip: 10.255.255.254:9000
=>Info: using ip-port socket
[UE] Connected via UnrealCV at 127.0.0.1:9000
OK: UnrealCV connected (Level PIE)


In [3]:
# 角の目印にビューを合わせてからこのセルを実行（必要な回数だけ繰り返し）
snapshots = lcp.list_camera_snapshots(ucv)
lcp.print_camera_report(snapshots)
lcp.export_camera_snapshots(snapshots)

if snapshots:
    cam = snapshots[0]
    print("\n[Handoff example — camera id=0]")
    print(lcp.format_corner_handoff(cam))
    print("\nTip: 別 id を使う場合は snapshots[camera_id] を指定してください。")

[Camera] count=1
  id= 0 PawnSensor               loc_cm=(   6272.47,    1164.16,    6488.92) loc_m=(  62.725,   11.642,   64.889) rot=(  -6.45,  174.76,    0.00)
[Camera] exported /home/winder17wsl_ishizawalab/00_kotaprivate/Program/SimWorld/dev/grid_env_level_semantic/.level_camera_snapshot.json

[Handoff example — camera id=0]
corner_xy_m=(62.7247, 11.6416)  # camera id=0 PawnSensor
corner_z_cm=6488.92  # sight height reference only

Tip: 別 id を使う場合は snapshots[camera_id] を指定してください。


In [ ]:
# 任意: 特定カメラ id の座標だけ表示（例: id=0 がプレイヤー視点のとき）
CAMERA_ID = 0

snapshots = lcp.list_camera_snapshots(ucv)
by_id = {s.camera_id: s for s in snapshots}
if CAMERA_ID not in by_id:
    raise KeyError(f"camera id={CAMERA_ID} not found; available={sorted(by_id)}")

cam = by_id[CAMERA_ID]
print(lcp.format_corner_handoff(cam))
x_m, y_m, z_m = lcp.vec3_to_m(cam.location_cm)
print(f"copy: x_m={x_m:.4f}, y_m={y_m:.4f}, z_m={z_m:.4f}")